---
short_title: Export nach BibTeX
numbering:
    heading_1: true
    heading_2: true
    title: true
---
# Export nach BibTeX

```{caution} Work In Progress
Diese Inhalte sind noch in Bearbeitung.
```

{term}`BibTeX` ist ein Dateiformat für Literaturverzeichnisse, das vor allem mit
LaTeX, aber auch mit vielen Literaturverwaltungsprogrammen genutzt wird. Eine
`.bib`-Datei besteht aus **Einträgen**, die mit `@typ{schlüssel, …}` beginnen.
Für Bücher nutzt man üblicherweise `@book`, für Zeitschriftenaufsätze `@article`.

In diesem Notebook lesen wir die {term}`MARC-XML`-Daten ein und erzeugen daraus für
jeden Titel einen BibTeX-Eintrag.


## MARC-Daten einlesen

Wir lesen die synthetischen Beispieldaten ein und wandeln sie – wie im
[Kapitel zum Einlesen](./020-Einlesen_Filtern.ipynb) – in Dictionaries um.


In [1]:
from pathlib import Path
from xml.etree import ElementTree as ET

MARC_NS = "http://www.loc.gov/MARC21/slim"
NAMENSRAUM = {"m": MARC_NS}


def finde_projektwurzel():
    """Sucht das Verzeichnis, das den Ordner assets/data enthält."""
    start = Path.cwd().resolve()
    for kandidat in [start, *start.parents]:
        if (kandidat / "assets" / "data").is_dir():
            return kandidat
    raise FileNotFoundError("Projektwurzel mit assets/data nicht gefunden.")


PROJEKTWURZEL = finde_projektwurzel()
DATEN = PROJEKTWURZEL / "assets" / "data"

records = ET.parse(DATEN / "beispieldaten.marcxml").getroot().findall("m:record", NAMENSRAUM)


def kontrolle(record, tag):
    feld = record.find(f"m:controlfield[@tag='{tag}']", NAMENSRAUM)
    return feld.text if feld is not None else None


def unterfeld(record, tag, code):
    feld = record.find(f"m:datafield[@tag='{tag}']", NAMENSRAUM)
    if feld is None:
        return None
    sub = feld.find(f"m:subfield[@code='{code}']", NAMENSRAUM)
    return sub.text if sub is not None else None


def alle_unterfelder(record, tag, code):
    return [
        sub.text
        for feld in record.findall(f"m:datafield[@tag='{tag}']", NAMENSRAUM)
        for sub in feld.findall(f"m:subfield[@code='{code}']", NAMENSRAUM)
    ]


def als_dict(record):
    return {
        "id": kontrolle(record, "001"),
        "isbn": unterfeld(record, "020", "a"),
        "autor": unterfeld(record, "100", "a"),
        "titel": unterfeld(record, "245", "a"),
        "untertitel": unterfeld(record, "245", "b"),
        "ort": unterfeld(record, "264", "a"),
        "verlag": unterfeld(record, "264", "b"),
        "jahr": unterfeld(record, "264", "c"),
        "schlagwoerter": alle_unterfelder(record, "650", "a"),
    }


titel_liste = [als_dict(record) for record in records]
print(f"{len(titel_liste)} Titel eingelesen")

18 Titel eingelesen


## Bibliografische Angaben für BibTeX

BibTeX erwartet Feldnamen wie `author`, `title` und `year`. Das Erscheinungsjahr
liefert MARC in `264 $c`, die ISBN in `020 $a`. Für die Autorenangabe nutzen wir
das MARC-Format „Nachname, Vorname“ direkt – BibTeX versteht dieses Format.

Ein BibTeX-Eintrag besteht aus:

- einem **Typ** (`@book`, `@article`, …),
- einem **Schlüssel** (eindeutig innerhalb der Datei) und
- den **Feldern** (`author = {…}`).


## Einen eindeutigen Schlüssel erzeugen

Als Schlüssel kombinieren wir den Nachnamen, das Jahr und das erste Wort des
Titels, z. B. `keller2021datenkuration`. Sonderzeichen entfernen wir, damit der
Schlüssel in LaTeX problemlos funktioniert.


In [2]:
import re


def bibtex_schluessel(eintrag):
    """Erzeugt einen eindeutigen BibTeX-Schlüssel."""
    nachname = eintrag["autor"].split(",")[0].lower()
    nachname = re.sub(r"[^a-z0-9]+", "", nachname)
    erstes_wort = eintrag["titel"].split()[0].lower()
    erstes_wort = re.sub(r"[^a-z0-9]+", "", erstes_wort)
    return f"{nachname}{eintrag['jahr']}{erstes_wort}"


for eintrag in titel_liste[:5]:
    print(bibtex_schluessel(eintrag))

keller2021datenkuration
okafor2019metadata
dupont2020catalogage
schneider2022praktische
almeida2018preserving


## Sonderzeichen für BibTeX maskieren

In BibTeX haben manche Zeichen eine Sonderbedeutung. Ein `&` muss als `\&`
geschrieben werden, sonst bricht der LaTeX-Lauf ab. Eine kleine Funktion
maskiert die wichtigsten Zeichen.


In [3]:
def maskiere(text):
    """Maskiert Sonderzeichen für BibTeX."""
    for zeichen in "&%$#_{}":
        text = text.replace(zeichen, "\\" + zeichen)
    return text


print(maskiere("Code & Buch"))

Code \& Buch


## Einen BibTeX-Eintrag erzeugen

Nun setzen wir die Bausteine zusammen. Der Titel wird mit dem Untertitel
verbunden, die Schlagwörter werden als `keywords` ausgegeben.


In [4]:
def als_bibtex(eintrag):
    """Erzeugt einen BibTeX-Eintrag vom Typ @book."""
    titel = eintrag["titel"].rstrip(" :")
    if eintrag["untertitel"]:
        titel = f"{titel}: {eintrag['untertitel']}"

    felder = [
        ("author", eintrag["autor"]),
        ("title", titel),
        ("year", eintrag["jahr"]),
        ("publisher", eintrag["verlag"]),
        ("address", eintrag["ort"]),
        ("isbn", eintrag["isbn"]),
        ("keywords", ", ".join(eintrag["schlagwoerter"])),
    ]

    zeilen = [f"@book{{{bibtex_schluessel(eintrag)},"]
    for name, wert in felder:
        zeilen.append(f"  {name} = {{{maskiere(str(wert))}}},")
    zeilen.append("}")
    return "\n".join(zeilen)


print(als_bibtex(titel_liste[0]))

@book{keller2021datenkuration,
  author = {Keller, Miriam},
  title = {Datenkuration in wissenschaftlichen Bibliotheken: Praktiken, Werkzeuge und Standards},
  year = {2021},
  publisher = {Wissenschaftsverlag Berlin},
  address = {Berlin},
  isbn = {9783110470017},
  keywords = {Bibliothekswissenschaft, Forschungsdaten, Metadaten},
}


## Die `.bib`-Datei schreiben

Zum Schluss schreiben wir alle Einträge, getrennt durch eine Leerzeile, in die
Datei `beispieldaten.bib`.


In [5]:
eintraege = [als_bibtex(eintrag) for eintrag in titel_liste]
bibtex = "\n\n".join(eintraege) + "\n"

ziel = DATEN / "beispieldaten.bib"
ziel.write_text(bibtex, encoding="utf-8")
print("geschrieben:", ziel.relative_to(PROJEKTWURZEL))
print(f"Einträge: {len(eintraege)}")

geschrieben: assets/data/beispieldaten.bib
Einträge: 18


## Kontrolle

Wir lesen die Datei zurück und zählen die BibTeX-Einträge. Jeder Eintrag beginnt
mit einem `@`.


In [6]:
inhalt = ziel.read_text(encoding="utf-8")
print("Anzahl @book-Einträge:", inhalt.count("@book"))

print("\n--- Auszug ---")
print("\n".join(inhalt.splitlines()[:12]))

Anzahl @book-Einträge: 18

--- Auszug ---
@book{keller2021datenkuration,
  author = {Keller, Miriam},
  title = {Datenkuration in wissenschaftlichen Bibliotheken: Praktiken, Werkzeuge und Standards},
  year = {2021},
  publisher = {Wissenschaftsverlag Berlin},
  address = {Berlin},
  isbn = {9783110470017},
  keywords = {Bibliothekswissenschaft, Forschungsdaten, Metadaten},
}

@book{okafor2019metadata,
  author = {Okafor, Chidi},


## Zusammenfassung

Sie haben MARC-Daten gelesen, auf BibTeX abgebildet und eine `.bib`-Datei
geschrieben. Damit ist der Transformationsweg dieses Moduls vollständig:
JSON → MARC-XML → Auswertung → BibTeX. Nun sind Sie an der Reihe: In der
[Aufgabe](./050-Aufgabe.md) wenden Sie das Gelernte selbstständig an.


## Selbsttest

Prüfen Sie zum Abschluss des Moduls Ihr Verständnis. Wählen Sie jeweils die passende Antwort aus.

In [ ]:
from jupyterquiz import display_quiz
c = { '--jq-multiple-choice-bg': '#202080',
      '--jq-mc-button-bg': '#fafafa',
      '--jq-mc-button-border': '#e0e0e0e0',
      '--jq-mc-button-inset-shadow': '#555555',
      '--jq-many-choice-bg': '#202080',
      '--jq-numeric-bg': '#202080',
      '--jq-numeric-input-bg': '#c0c0c0',
      '--jq-numeric-input-label': '#101010',
      '--jq-numeric-input-shadow': '#999999',
      '--jq-string-bg': '#202080',
      '--jq-incorrect-color': '#c80202',
      '--jq-correct-color': '#009113',
      '--jq-text-color': '#fafafa',
      '--jq-link-color': '#9abafa' }


In [ ]:
q = [{
        "question": "Mit welchem Modul der Standardbibliothek liest der Kurs XML-Dateien ein?",
        "type": "multiple_choice",
        "answers": [
            { "code": "xml.etree.ElementTree", "correct": True },
            { "code": "pymarc", "correct": False },
            { "code": "csv", "correct": False },
            { "code": "json", "correct": False }
        ]
    },
    {
        "question": "In welchem MARC-Feld stehen Titel und Untertitel?",
        "type": "multiple_choice",
        "answers": [
            { "code": "245", "correct": True },
            { "code": "020", "correct": False },
            { "code": "100", "correct": False },
            { "code": "264", "correct": False }
        ]
    },
    {
        "question": "Welches Kontrollfeld kennzeichnet einen Record eindeutig?",
        "type": "multiple_choice",
        "answers": [
            { "code": "001", "correct": True },
            { "code": "003", "correct": False },
            { "code": "008", "correct": False },
            { "code": "650", "correct": False }
        ]
    },
    {
        "question": "In welchem Unterfeld steht die ISBN?",
        "type": "multiple_choice",
        "answers": [
            { "code": "020 $a", "correct": True },
            { "code": "041 $a", "correct": False },
            { "code": "264 $c", "correct": False },
            { "code": "650 $a", "correct": False }
        ]
    },
    {
        "question": "Wie viele Records enthält <code>assets/data/beispieldaten.marcxml</code>?",
        "type": "string",
        "answers": [
            { "answer": "18", "correct": True,
              "feedback": "Richtig. Die Datei enthält 18 Records." }
        ]
    }]
display_quiz(q, colors=c)
